# Absorption v2: fixing the metric and testing the rho-below-class-frequency mechanism

The first absorption run failed its own prediction (TopK ~0 absorption, RateKL nonzero, CIFAR both ~0.5) for three diagnosed reasons, each fixed here:

1. **TopK's ~0 absorption was mostly a metric artifact.** Max-F1 latent selection picks always-on hub latents as "main" when nothing else tracks a class well -- and an always-on latent can never register an absorption instance (it never goes silent). Fixed: `main_fire_rate_overall >= 0.5` is flagged as `is_hub_main`; those classes are excluded from the headline means, and `n_hub_main_classes` is reported so a fully-hub-contaminated method shows as *unreliable*, not falsely clean.
2. **RateKL's CIFAR absorption (~0.54, ~= its own miss rate) suggested the compensation criterion passes near-trivially on weak CIFAR pixel probes.** Fixed: a `null_absorption_rate` control (same computation on random latent sets) -- `absorption_above_null` is the number to trust now, not raw `absorption_rate`.
3. **RateKL's own two-sided `KL(rho||p_hat_f)` may manufacture absorption**: any concept whose natural frequency exceeds rho (class base rate 0.10 > rho=0.09) gets forced to drop ~10% of its instances to hit the target -- exactly the holes-with-compensation structure absorption consists of. Two tests: (a) `rho=0.12` (above class frequency) with the *original* two-sided objective, (b) a new one-sided cap+budget objective (`rate_cap_budget_sae.py`) that only penalizes firing rates *above* a cap and holds the sparsity budget separately, removing the sub-rho pull entirely.

**What to look for:** if the mechanism diagnosis is right, `rho=0.12` and `RateCapBudget` should both show lower `mean_absorption_above_null` than the `rho=0.09` baseline, with `mean_main_f1` holding up (not just avoiding absorption by tracking classes worse). Also check `n_hub_main_classes` for TopK -- if it's high, its absorption number (whatever it is) isn't trustworthy and should be reported as such, not as a clean win.

**Before running:** Runtime -> Change runtime type -> GPU.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (go to Runtime > Change runtime type > GPU)')

In [ ]:
!git clone https://github.com/willkn/SAE-Gini.git
%cd SAE-Gini/experiments

In [ ]:
# Each (dataset, seed) trains 5 models: RateKL rho=0.09 (baseline, two-sided),
# RateKL rho=0.12 (mechanism test, two-sided), RateCapBudget (one-sided fix),
# TopK, TunedL1 -- all scored against the same freshly-fit class probes.
for ds in ['fashion_mnist', 'mnist', 'cifar10']:
    for seed in [0, 1, 2]:
        !python absorption_v2_experiment.py --dataset {ds} --seed {seed} --rho-low 0.09 --rho-high 0.12 --rho-max 0.15

In [ ]:
import json, glob, statistics
from collections import defaultdict

agg = defaultdict(lambda: defaultdict(list))
for path in sorted(glob.glob('results/absorption_v2/*_seed*.json')):
    ds = path.split('/')[-1].rsplit('_seed', 1)[0]
    with open(path) as f:
        for model, r in json.load(f).items():
            agg[ds][model].append(r)

for ds, models in agg.items():
    print(f"\n-- {ds} --")
    print(f"{'Model':28s} {'Absorption':>13s} {'vsNull':>13s} {'MainF1':>13s} {'HubMainClasses':>15s}")
    for model, runs in models.items():
        def ms(key):
            vals = [r[key] for r in runs if r[key] == r[key]]  # drop NaN
            if not vals:
                return 'all-NaN'
            s = statistics.stdev(vals) if len(vals) > 1 else 0.0
            return f"{statistics.mean(vals):.4f}+/-{s:.4f}"
        hub_counts = [r['n_hub_main_classes'] for r in runs]
        hub_str = f"{statistics.mean(hub_counts):.1f}/10"
        print(f"{model:28s} {ms('mean_absorption_rate'):>13s} {ms('mean_absorption_above_null'):>13s} "
              f"{ms('mean_main_f1'):>13s} {hub_str:>15s}")

In [ ]:
!zip -r absorption_v2_results.zip results/absorption_v2
from google.colab import files
files.download('absorption_v2_results.zip')